# Dat 200 CA5 Group 4
Lasse Lindholm and Brage Bromset Bestvold

## Imports

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns


# Load data

In [ ]:
train = pd.read_csv("trainCA5.csv")
test = pd.read_csv("testCA5.csv")
train.head()

: 

: 

In [ ]:
# As shown in the head, there are rows with missing data, so we remove these
train = train.dropna()

: 

: 

## Data exploration


The task description tells us what the data in reality should look like, which should allow us to clean it based off of the data exploration data

id: Identifier, carries no information and must not be used as a feature
age: ages are 18-90
bps: should be between 60 and 280 (source: Nurse student who worked as "helsefagarbeider" for many years and has "seen it all") Influenced by age, diet, stress and exercise within the simulation.

sleep duration avg: negatively impacted by stress
sleep wuality: self rated, 1-10
cardio min per week: total minutes of elevated heart rate exercise
water intake liters: average daily water consumtion, includes a "noise" variable
screen time rec hrs: daily ours spent on non-work screens
social media hrs: daily hours spent on some

diet ultraprosessed: percentage of diet that is ultraprocessed
diet omega 3 frq: days per week consuming "brain-healthy" food
smoking status: current, former or never
alcohol drinks wk: total alcoholic beverages per week

novel learning hours wk: hours per week spent learning a new difficult skill
social connection score: self rated score from 1-10
chronic stress score: self rated scale from 1-10

All of this should be explored here, and taken into account in the data cleaning stage.



### Correlation heatmap

In [ ]:
# Modified correlation heatmap taken from the 04_correlationmap.ipynb example from canvas
# https://nmbu.instructure.com/courses/14056/files/folder/Lecture_12_Preprocessing?preview=2814845
def plot_correlation_heatmap(data, title, method="pearson"):
    """
    Plot a correlation heatmap (lower triangle only) for the numeric columns of a dataset.
    NaNs are handled pairwise by pandas, so missing values do not break it.
    """
    corr = data.corr(numeric_only=True, method=method)
    mask = np.triu(np.ones_like(corr, dtype=bool))  # hide the mirrored upper triangle
    plt.figure(figsize=(14, 11))
    sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
                vmin=-1, vmax=1, square=True, linewidths=0.5)
    plt.title(title)
    plt.show()

heatmap_data = train.drop(columns="id").copy()
heatmap_data["smoking_status"] = heatmap_data["smoking_status"].map({"Never": 0, "Former": 1, "Current": 2})

plot_correlation_heatmap(heatmap_data, "Correlation heatmap of the training data")

: 

: 

None of the values seem to have very strong corelations, the plan for now is to keep all

### Box plots


In [ ]:
# Only numeric columns can be box plotted (smoking_status is text, id carries no information)
num_cols = train.drop(columns="id").select_dtypes(include="number").columns

n_cols = int(np.ceil(len(num_cols) / 2))
fig, axes = plt.subplots(2, n_cols, figsize=(30, 8))
fig.suptitle("Boxplots of each numeric column", fontsize=18)

for ax, col in zip(axes.flatten(), num_cols):
    ax.boxplot(train[col].dropna())  # NaNs must be dropped
    ax.set_title(col, fontsize=9)
    ax.set_xticks([])

# hide the unused axes if there is an odd number of columns
for ax in axes.flatten()[len(num_cols):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

: 

: 

The box plots reveal that every single column appear to have a little bit of random nonsense numbers thrown in. A very easy example is the age column whics's box plot shows a lot of dots around the 200 year mark which is just ridiculous, as well as negative 100 years old.

## Data cleaning

In [ ]:
# clamp all features to sensible ranges for further analysis
train = train[train['age'].between(18,90)]
train = train[train['sleep_duration_hrs'] >= 0]
train = train[train['sleep_quality_score'].between(1,10)]
train = train[train['cardio_mins_wk'] >= 0]
train = train[train['water_intake_liters'] >= 0]
train = train[train['screen_time_recreational_hrs'].between(0,24)]
train = train[train['social_media_hrs'].between(0,24)]
train = train[train['blood_pressure_sys'].between(60,280)]
train = train[train['diet_ultra_processed_pct'].between(0,100)]
train = train[train['diet_omega3_freq'].between(0,7)]
train = train[train['alcohol_drinks_wk'] >= 0]
train = train[train['novel_learning_hrs_wk'] >= 0]
train = train[train['social_connection_score'].between(1,10)]
train = train[train['chronic_stress_score'].between(1,10)]
train

: 

: 

After cleaning missing data and removing outliers and unrealistic scores only 100067 data entires are left, which is less than a third of the original dataset. 

In [ ]:
# Modified correlation heatmap taken from the 04_correlationmap.ipynb example from canvas
# https://nmbu.instructure.com/courses/14056/files/folder/Lecture_12_Preprocessing?preview=2814845
def plot_correlation_heatmap(data, title, method="pearson"):
    """
    Plot a correlation heatmap (lower triangle only) for the numeric columns of a dataset.
    NaNs are handled pairwise by pandas, so missing values do not break it.
    """
    corr = data.corr(numeric_only=True, method=method)
    mask = np.triu(np.ones_like(corr, dtype=bool))  # hide the mirrored upper triangle
    plt.figure(figsize=(14, 11))
    sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
                vmin=-1, vmax=1, square=True, linewidths=0.5)
    plt.title(title)
    plt.show()

# Id is dropped, smoking status has text so is removed for this heatmap
# Could maybe try the state trick from last CA to have 3 seperate columns for each
heatmap_data = train.drop(columns="id").copy()
heatmap_data["smoking_status"] = heatmap_data["smoking_status"].map({"Never": 0, "Former": 1, "Current": 2})

plot_correlation_heatmap(heatmap_data, "Correlation heatmap of the training data after cleaning")

: 

: 

In [ ]:
# Only numeric columns can be box plotted (smoking_status is text, id carries no information)
num_cols = train.drop(columns="id").select_dtypes(include="number").columns

n_cols = int(np.ceil(len(num_cols) / 2))
fig, axes = plt.subplots(2, n_cols, figsize=(30, 8))
fig.suptitle("Boxplots of each numeric column after cleaning", fontsize=18)

for ax, col in zip(axes.flatten(), num_cols):
    ax.boxplot(train[col].dropna())  # NaNs must be dropped
    ax.set_title(col, fontsize=9)
    ax.set_xticks([])

# hide the unused axes if there is an odd number of columns
for ax in axes.flatten()[len(num_cols):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

: 

: 

When looking at the heatmap and the boxplot as well as taking the information about each individual feature into consideration, some features do not correlate at all with target and can be removed. When looking at the boxplots, because removing unplausible (and impossible values) the data looks realistic, for example the cardio minutes per week has a long tail, which supports few outliers who exercise a lot more than most. Because of the high number of outliers some of them may be considered to be removed, but we deem the variance to be realistic in a large group of humans, e.g alcohol intake and novel learning where it is realistic that most don't drink that much, but some do.

## preprocessing

## Model Evaluation